# Audit #1: Page Migration Completeness

This script takes the live production XML file and converts the <loc>s to a simple url list for http response code 200 testing via Screaming Frog. There is a fallback in the event of rate limiting or 403 timeouts.

In [6]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from urllib.parse import urlparse, urlunparse, urljoin
from datetime import datetime

# ============================================================
# CONFIGURATION
# ============================================================

PRODUCTION_SITEMAP = "https://www.varsityzone.com/sitemap.xml"
PRODUCTION_DOMAIN = "www.varsityzone.com"
STAGING_DOMAIN = "varsityzonedev.wpenginepowered.com"

# Saved XML file, in the same directory as the notebook
FALLBACK_SITEMAP = Path("scorpion_xml_sitemap.xml")

OUTPUT_FILE = (
    f"vz-page-migration-expected-urls-"
    f"{datetime.now():%Y-%m-%d}.csv"
)

TIMEOUT = 30

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; VZMigrationAudit/1.0)"
})

visited_sitemaps = set()
production_urls = set()

sitemap_source = "LIVE"
fallback_used = False


# ============================================================
# XML HELPERS
# ============================================================

def tag_name(element):
    """Remove XML namespace from an element name."""
    return element.tag.split("}")[-1]


def load_local_sitemap():
    """Read the saved Scorpion XML sitemap."""

    if not FALLBACK_SITEMAP.is_file():
        raise FileNotFoundError(
            f"Fallback sitemap not found: "
            f"{FALLBACK_SITEMAP.resolve()}"
        )

    print(f"Using local fallback: {FALLBACK_SITEMAP.resolve()}")

    return ET.parse(FALLBACK_SITEMAP).getroot()


def get_xml(url, allow_local_fallback=False):
    """
    Download sitemap XML.

    If the live production sitemap returns HTTP 403,
    use the local XML fallback.
    """

    global sitemap_source, fallback_used

    print(f"Requesting: {url}")

    response = session.get(url, timeout=TIMEOUT)

    if response.status_code == 403 and allow_local_fallback:
        print("HTTP 403: Access forbidden.")
        print("Switching to saved XML sitemap.")

        sitemap_source = "LOCAL_FALLBACK"
        fallback_used = True

        return load_local_sitemap()

    # Other HTTP errors should not silently pass.
    response.raise_for_status()

    return ET.fromstring(response.content)


# ============================================================
# SITEMAP PARSER
# ============================================================

def parse_sitemap(sitemap_url, is_primary=False):
    """
    Supports both:
      - <urlset> (standard sitemap)
      - <sitemapindex> (nested sitemaps)
    """

    if sitemap_url in visited_sitemaps:
        return

    visited_sitemaps.add(sitemap_url)

    root = get_xml(
        sitemap_url,
        allow_local_fallback=is_primary
    )

    root_type = tag_name(root)

    if root_type == "sitemapindex":

        for sitemap in root:

            if tag_name(sitemap) != "sitemap":
                continue

            for child in sitemap:

                if tag_name(child) == "loc" and child.text:

                    nested_url = urljoin(
                        sitemap_url,
                        child.text.strip()
                    )

                    parse_sitemap(nested_url)

    elif root_type == "urlset":

        for item in root:

            if tag_name(item) != "url":
                continue

            for child in item:

                if tag_name(child) == "loc" and child.text:

                    production_urls.add(
                        child.text.strip()
                    )

    else:
        raise ValueError(
            f"Unexpected XML root element: {root_type}"
        )


# ============================================================
# URL CONVERSION
# ============================================================

def convert_to_staging(production_url):
    """
    Replace production domain with staging domain.

    Remove trailing slash from paths except root.
    Preserve URL path and query string.
    """

    parsed = urlparse(production_url)

    if parsed.hostname not in (
        PRODUCTION_DOMAIN,
        "varsityzone.com"
    ):
        raise ValueError(
            f"Unexpected production domain: {production_url}"
        )

    path = parsed.path.rstrip("/") or "/"

    return urlunparse((
        "https",
        STAGING_DOMAIN,
        path,
        "",
        parsed.query,
        ""
    ))


# ============================================================
# EXECUTE
# ============================================================

print("=" * 60)
print("VARSITY ZONE - PAGE MIGRATION AUDIT #1")
print("=" * 60)

parse_sitemap(
    PRODUCTION_SITEMAP,
    is_primary=True
)

rows = []

for production_url in sorted(production_urls):

    rows.append({
        "production_url": production_url,
        "expected_staging_url": convert_to_staging(
            production_url
        )
    })

df = pd.DataFrame(
    rows,
    columns=[
        "production_url",
        "expected_staging_url"
    ]
)

if df.empty:
    raise RuntimeError(
        "No production URLs found. CSV not generated."
    )

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# AUDIT SUMMARY
# ============================================================

print("\n=== MIGRATION AUDIT SUMMARY ===")
print(f"Sitemap source: {sitemap_source}")
print(f"Local fallback used: {fallback_used}")
print(f"Sitemaps processed: {len(visited_sitemaps)}")
print(f"Production URLs found: {len(df)}")
print(f"Expected staging URLs: {len(df)}")
print(f"CSV saved: {OUTPUT_FILE}")

if fallback_used:
    print(
        "\nNOTE: Inventory generated from saved XML snapshot, "
        "not the current live sitemap."
    )

display(df.head(20))


VARSITY ZONE - PAGE MIGRATION AUDIT #1
Requesting: https://www.varsityzone.com/sitemap.xml
HTTP 403: Access forbidden.
Switching to saved XML sitemap.
Using local fallback: C:\Users\luke_\Documents\Python Scripts\FranchiseRocket\VARSITY ZONE MIGRATION FRAMEWORK NOV 2026\scorpion_xml_sitemap.xml

=== MIGRATION AUDIT SUMMARY ===
Sitemap source: LOCAL_FALLBACK
Local fallback used: True
Sitemaps processed: 1
Production URLs found: 1253
Expected staging URLs: 1253
CSV saved: vz-page-migration-expected-urls-2026-11-09.csv

NOTE: Inventory generated from saved XML snapshot, not the current live sitemap.


,production_url,expected_staging_url
0,https://www.varsityzone.com/,https://varsityzonedev.wpenginepowered.com/
1,https://www.varsityzone.com/about-us/,https://varsityzonedev.wpenginepowered.com/abo...
2,https://www.varsityzone.com/about-us/blog/,https://varsityzonedev.wpenginepowered.com/abo...
3,https://www.varsityzone.com/about-us/blog/cate...,https://varsityzonedev.wpenginepowered.com/abo...
4,https://www.varsityzone.com/about-us/blog/cate...,https://varsityzonedev.wpenginepowered.com/abo...
5,https://www.varsityzone.com/about-us/blog/cate...,https://varsityzonedev.wpenginepowered.com/abo...
6,https://www.varsityzone.com/about-us/blog/cate...,https://varsityzonedev.wpenginepowered.com/abo...
7,https://www.varsityzone.com/about-us/blog/cate...,https://varsityzonedev.wpenginepowered.com/abo...
8,https://www.varsityzone.com/about-us/blog/cate...,https://varsityzonedev.wpenginepowered.com/abo...
9,https://www.varsityzone.com/about-us/blog/cate...,https://varsityzonedev.wpenginepowered.com/abo...


## Audit #1 — Screaming Frog response code exception report

In [9]:

# ============================================================
# AUDIT #1 - SCREAMING FROG RESPONSE CODE EXCEPTION REPORT
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

INPUT_FILE = Path(
    "staging-9-nov-listmode-internal-audit1-response-codes.xlsx"
)

OUTPUT_FILE = Path(
    f"vz-audit1-response-code-exceptions-"
    f"{datetime.now():%Y-%m-%d}.xlsx"
)

# ------------------------------------------------------------
# LOAD SCREAMING FROG EXPORT
# ------------------------------------------------------------

if not INPUT_FILE.is_file():
    raise FileNotFoundError(
        f"Screaming Frog export not found: {INPUT_FILE.resolve()}"
    )

sf = pd.read_excel(INPUT_FILE, engine="openpyxl")

# Clean column headings
sf.columns = sf.columns.astype(str).str.strip()

print("Screaming Frog columns:", sf.columns.tolist())

# ------------------------------------------------------------
# IDENTIFY REQUIRED COLUMNS
# ------------------------------------------------------------

def find_column(candidates, required=True):
    lookup = {col.lower(): col for col in sf.columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    if required:
        raise ValueError(
            f"Required column missing. Expected one of: {candidates}"
        )

    return None


url_col = find_column(["Address", "URL"])
status_col = find_column(["Status Code", "HTTP Status Code"])

status_text_col = find_column(
    ["Status", "Status Description"],
    required=False
)

redirect_col = find_column(
    ["Redirect URL", "Redirect Location"],
    required=False
)

# ------------------------------------------------------------
# NORMALIZE RESPONSE CODES
# ------------------------------------------------------------

sf["http_status_code"] = pd.to_numeric(
    sf[status_col],
    errors="coerce"
).astype("Int64")

# ------------------------------------------------------------
# CLASSIFY EXCEPTIONS
# ------------------------------------------------------------

def classify_exception(code):

    if pd.isna(code):
        return (
            "UNKNOWN",
            "No valid HTTP response code - investigate"
        )

    code = int(code)

    if code == 200:
        return ("PASS", "OK")

    if code == 404:
        return (
            "MISSING PAGE",
            "Not brought across - developers to investigate"
        )

    if code in (301, 302):
        return (
            "REDIRECT",
            "Redirects elsewhere - developers to investigate"
        )

    if code == 410:
        return (
            "MISSING PAGE",
            "Page gone - developers to investigate"
        )

    if 300 <= code < 400:
        return (
            "OTHER REDIRECT",
            "Unexpected redirect - developers to investigate"
        )

    if 400 <= code < 500:
        return (
            "CLIENT ERROR",
            "HTTP client error - developers to investigate"
        )

    if 500 <= code < 600:
        return (
            "SERVER ERROR",
            "HTTP server error - developers to investigate"
        )

    return (
        "OTHER NON-200",
        "Unexpected response - investigate"
    )


classifications = sf["http_status_code"].apply(classify_exception)

sf["audit_result"] = classifications.apply(lambda x: x[0])
sf["developer_action"] = classifications.apply(lambda x: x[1])

# ------------------------------------------------------------
# BUILD EXCEPTION REPORT
# ------------------------------------------------------------

exceptions = sf[sf["audit_result"] != "PASS"].copy()

report = pd.DataFrame({
    "staging_url": exceptions[url_col],
    "http_status_code": exceptions["http_status_code"],
    "audit_result": exceptions["audit_result"],
    "developer_action": exceptions["developer_action"]
})

if status_text_col:
    report["status_description"] = exceptions[status_text_col]

if redirect_col:
    report["redirect_destination"] = exceptions[redirect_col]

# Prioritize missing pages, then redirects, then other errors
priority = {
    "MISSING PAGE": 1,
    "REDIRECT": 2,
    "OTHER REDIRECT": 3,
    "SERVER ERROR": 4,
    "CLIENT ERROR": 5,
    "UNKNOWN": 6,
    "OTHER NON-200": 7
}

report["_priority"] = report["audit_result"].map(priority).fillna(99)

report = (
    report.sort_values(
        ["_priority", "http_status_code", "staging_url"],
        na_position="last"
    )
    .drop(columns="_priority")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# BUILD SUMMARY
# ------------------------------------------------------------

total = len(sf)
passed = (sf["audit_result"] == "PASS").sum()
failed = len(report)

summary = pd.DataFrame([
    {"Metric": "Total URLs checked", "Count": total},
    {"Metric": "HTTP 200 - Pass", "Count": passed},
    {"Metric": "Non-200 exceptions", "Count": failed},
    {"Metric": "HTTP 404 - Missing", "Count": (sf["http_status_code"] == 404).sum()},
    {"Metric": "HTTP 301 - Redirect", "Count": (sf["http_status_code"] == 301).sum()},
    {"Metric": "HTTP 302 - Redirect", "Count": (sf["http_status_code"] == 302).sum()},
    {"Metric": "Other non-200", "Count": (
        (~sf["http_status_code"].isin([200, 301, 302, 404]))
    ).sum()}
])

# ------------------------------------------------------------
# EXPORT EXCEL
# ------------------------------------------------------------

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    report.to_excel(
        writer,
        sheet_name="All Exceptions",
        index=False
    )

    report[report["http_status_code"] == 404].to_excel(
        writer,
        sheet_name="404 Missing",
        index=False
    )

    report[report["http_status_code"].isin([301, 302])].to_excel(
        writer,
        sheet_name="301-302 Redirects",
        index=False
    )

    report[
        ~report["http_status_code"].isin([301, 302, 404])
    ].to_excel(
        writer,
        sheet_name="Other Exceptions",
        index=False
    )

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VARSITY ZONE - MIGRATION AUDIT #1")
print("SCREAMING FROG RESPONSE CODE ANALYSIS")
print("=" * 60)

print(f"Total URLs checked: {total}")
print(f"HTTP 200 passes: {passed}")
print(f"Non-200 exceptions: {failed}")
print(f"404 missing pages: {(sf['http_status_code'] == 404).sum()}")
print(f"301 redirects: {(sf['http_status_code'] == 301).sum()}")
print(f"302 redirects: {(sf['http_status_code'] == 302).sum()}")
print(f"\nException report saved: {OUTPUT_FILE.resolve()}")

display(summary)
display(report.head(30))


C:\Users\luke_\anaconda3\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Screaming Frog columns: ['Address', 'Content Type', 'Status Code', 'Status', 'Indexability', 'Indexability Status', 'Title 1', 'Title 1 Length', 'Title 1 Pixel Width', 'Meta Description 1', 'Meta Description 1 Length', 'Meta Description 1 Pixel Width', 'Meta Keywords 1', 'Meta Keywords 1 Length', 'H1-1', 'H1-1 Length', 'H1-2', 'H1-2 Length', 'H2-1', 'H2-1 Length', 'H2-2', 'H2-2 Length', 'Meta Robots 1', 'X-Robots-Tag 1', 'Meta Refresh 1', 'Canonical Link Element 1', 'rel="next" 1', 'rel="prev" 1', 'HTTP rel="next" 1', 'HTTP rel="prev" 1', 'amphtml Link Element', 'Size (Bytes)', 'Transferred (Bytes)', 'Total Transferred (Bytes)', 'CO2 (mg)', 'Carbon Rating', 'Word Count', 'Sentence Count', 'Average Words Per Sentence', 'Flesch Reading Ease Score', 'Readability', 'Text Ratio', 'Crawl Depth', 'Folder Depth', 'Link Score', 'Inlinks', 'Unique Inlinks', 'Unique JS Inlinks', '% of Total', 'Outlinks', 'Unique Outlinks', 'Unique JS Outlinks', 'External Outlinks', 'Unique External Outlinks', 'Un

,Metric,Count
0,Total URLs checked,1262
1,HTTP 200 - Pass,1217
2,Non-200 exceptions,45
3,HTTP 404 - Missing,29
4,HTTP 301 - Redirect,13
5,HTTP 302 - Redirect,3
6,Other non-200,0


,staging_url,http_status_code,audit_result,developer_action,status_description,redirect_destination
0,https://varsityzonedev.wpenginepowered.com/arv...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
1,https://varsityzonedev.wpenginepowered.com/cle...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
2,https://varsityzonedev.wpenginepowered.com/fis...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
3,https://varsityzonedev.wpenginepowered.com/fra...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
4,https://varsityzonedev.wpenginepowered.com/fra...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
5,https://varsityzonedev.wpenginepowered.com/fri...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
6,https://varsityzonedev.wpenginepowered.com/gal...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
7,https://varsityzonedev.wpenginepowered.com/lov...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
8,https://varsityzonedev.wpenginepowered.com/moo...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
9,https://varsityzonedev.wpenginepowered.com/nor...,404,MISSING PAGE,Not brought across - developers to investigate,Not Found,NaN
